# Flavor Expansion Walkthrough

This notebook demonstrates the new flavor-index and class-member expansion features.

It uses only the `Model(..., lagrangian_decl=...)` API and then extracts rules from `model.lagrangian()`.

It covers:

1. explicit `IndexRole.FLAVOR` metadata,
2. generic fields with concrete class members,
3. compact vs expanded Feynman rules,
4. non-diagonal flavor tensors,
5. one-index diagonal tensors with `allow_summation=True`,
6. zero tensor components dropping expanded vertices,
7. gauge indices staying symbolic,
8. shared and independent flavor labels across field classes,
9. representative error messages.


In [1]:
import sys
from pathlib import Path
from fractions import Fraction

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from symbolica import S
from symbolica.community.spenso import Representation

from model import (
    COLOR_FUND_INDEX,
    Field,
    IndexRole,
    IndexType,
    Model,
    Parameter,
    SPINOR_INDEX,
    flavor_index,
)

print("Python:", sys.executable)
print("Repository root:", ROOT)


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
Repository root: /Users/rems/Documents/MScThesis


In [2]:
def show(title, result):
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", expression)
            print()
    else:
        print(result)
        print()


def dirac_field(name, *, indices=(), symbol=None, conjugate_symbol=None):
    if symbol is None:
        symbol = S(name)
    if conjugate_symbol is None:
        conjugate_symbol = S(f"{name}bar")
    return Field(
        name,
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=symbol,
        conjugate_symbol=conjugate_symbol,
        indices=indices,
    )


def scalar_field(name, *, symbol=None):
    if symbol is None:
        symbol = S(name)
    return Field(name, spin=0, self_conjugate=True, symbol=symbol)


def flavor_family(generic_name, member_names, flavor, *, extra_indices=()):
    members = tuple(
        dirac_field(
            member_name,
            indices=extra_indices + (SPINOR_INDEX,),
            symbol=S(member_name),
            conjugate_symbol=S(f"{member_name}bar"),
        )
        for member_name in member_names
    )
    generic = Field(
        generic_name,
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S(generic_name.lower()),
        conjugate_symbol=S(f"{generic_name.lower()}bar"),
        indices=(flavor,) + extra_indices + (SPINOR_INDEX,),
        flavor_index=flavor,
        class_members=members,
    )
    return generic, members


## 1. Define a flavor index and a generic field class

The generic field carries the flavor slot. Each class member drops that slot.


In [3]:
flavor = flavor_index("Flavor", 3)
psi, (e, mu, tau) = flavor_family("Psi", ("e", "mu", "tau"), flavor)
phi = scalar_field("Phi")

print("flavor.role:", flavor.role)
print("flavor.dimension:", flavor.dimension)
print("psi.indices:", [index.name for index in psi.indices])
print("psi.flavor_index_slot():", psi.flavor_index_slot())
print("members:", [member.name for member in psi.class_members])
print("member indices:", {member.name: [index.name for index in member.indices] for member in psi.class_members})
print("mapping:", {k: psi.class_member_for(k).name for k in range(1, flavor.dimension + 1)})


flavor.role: IndexRole.FLAVOR
flavor.dimension: 3
psi.indices: ['Flavor', 'Spinor']
psi.flavor_index_slot(): 0
members: ['e', 'mu', 'tau']
member indices: {'e': ['Spinor'], 'mu': ['Spinor'], 'tau': ['Spinor']}
mapping: {1: 'e', 2: 'mu', 3: 'tau'}


## 2. Compact vs expanded diagonal interaction

The compact rule keeps the generic field and a flavor identity tensor. The expanded rule produces only the diagonal member vertices.


In [4]:
f = S("f")
lag = Model(
    fields=(psi, phi),
    lagrangian_decl=S("lam") * psi.bar(f) * psi(f) * phi,
).lagrangian()

print("compact signatures", [signature.names for signature in lag.vertex_signatures()])
print("expanded signatures", [signature.names for signature in lag.vertex_signatures(flavor_expand=True)])
print()
print("compact rule Psi.bar, Psi, Phi")
print(lag.feynman_rule(psi.bar, psi, phi, simplify=True))
print()
print("expanded rule e.bar, e, Phi")
print(lag.feynman_rule(e.bar, e, phi, simplify=True, flavor_expand=True))
print()
print("off-diagonal e.bar, mu, Phi is absent")
try:
    print(lag.feynman_rule(e.bar, mu, phi, simplify=True, flavor_expand=True))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


compact signatures [('Psi.bar', 'Psi', 'Phi')]
expanded signatures [('e.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('tau.bar', 'tau', 'Phi')]

compact rule Psi.bar, Psi, Phi
1𝑖*lam*g(bis(4, i1),bis(4, i2))*g(cof(3, f1),cof(3, f2))

expanded rule e.bar, e, Phi
1𝑖*lam*g(bis(4, i1),bis(4, i2))

off-diagonal e.bar, mu, Phi is absent
ValueError: No matching interaction terms for: e.bar, mu, Phi.
Available signatures:
  - e.bar, e, Phi
  - mu.bar, mu, Phi
  - tau.bar, tau, Phi


## 3. Non-diagonal two-index flavor tensor

Independent flavor labels produce all member pairs.


In [5]:
f_mix = S("fMix")
g_mix = S("gMix")
Y = Parameter("Y", indices=(flavor, flavor))
mixed_model = Model(
    fields=(psi, phi),
    parameters=(Y,),
    lagrangian_decl=S("gY") * psi.bar(f_mix) * Y(f_mix, g_mix) * psi(g_mix) * phi,
)
mixed_L = mixed_model.lagrangian()

print("expanded signatures for Y[f, g]", [signature.names for signature in mixed_L.vertex_signatures(flavor_expand=True)])
print()
print("e.bar, mu, Phi")
print(mixed_L.feynman_rule(e.bar, mu, phi, simplify=True, flavor_expand=True))
print()
print("mu.bar, e, Phi")
print(mixed_L.feynman_rule(mu.bar, e, phi, simplify=True, flavor_expand=True))
print()
print("tau.bar, tau, Phi")
print(mixed_L.feynman_rule(tau.bar, tau, phi, simplify=True, flavor_expand=True))


expanded signatures for Y[f, g] [('e.bar', 'e', 'Phi'), ('e.bar', 'mu', 'Phi'), ('e.bar', 'tau', 'Phi'), ('mu.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('mu.bar', 'tau', 'Phi'), ('tau.bar', 'e', 'Phi'), ('tau.bar', 'mu', 'Phi'), ('tau.bar', 'tau', 'Phi')]

e.bar, mu, Phi
1𝑖*gY*g(bis(4, i1),bis(4, i2))*Y(1,2)

mu.bar, e, Phi
1𝑖*gY*g(bis(4, i1),bis(4, i2))*Y(2,1)

tau.bar, tau, Phi
1𝑖*gY*g(bis(4, i1),bis(4, i2))*Y(3,3)


## 4. One-index diagonal tensor and `allow_summation=True`

The shorthand `y[f] Psi.bar[f] Psi[f] Phi` is rejected unless the parameter explicitly opts in.


In [6]:
y_bad = Parameter("yBad", indices=(flavor,))
bad_model = Model(
    fields=(psi, phi),
    parameters=(y_bad,),
    lagrangian_decl=y_bad(f) * psi.bar(f) * psi(f) * phi,
)
print("without allow_summation")
try:
    bad_model.lagrangian().vertex_signatures(flavor_expand=True)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

y_diag = Parameter("yDiag", indices=(flavor,), allow_summation=True)
diag_model = Model(
    fields=(psi, phi),
    parameters=(y_diag,),
    lagrangian_decl=y_diag(f) * psi.bar(f) * psi(f) * phi,
)
diag_L = diag_model.lagrangian()

print("diagonal y[f] signatures", [signature.names for signature in diag_L.vertex_signatures(flavor_expand=True)])
print()
print("mu.bar, mu, Phi")
print(diag_L.feynman_rule(mu.bar, mu, phi, simplify=True, include_delta=True, flavor_expand=True).format(show_namespaces=False, color_top_level_sum=False, color_builtin_symbols=False, multiplication_operator='*'))


without allow_summation
ValueError: Parameter 'yBad' uses flavor label 'f' more than twice in one term. Mark the parameter with allow_summation=True to use this diagonal shorthand.

diagonal y[f] signatures [('e.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('tau.bar', 'tau', 'Phi')]

mu.bar, mu, Phi
1𝑖*(2*𝜋)^d*g(bis(4,i1),bis(4,i2))*Delta(q1+q2+q3)*yDiag(2)


## 5. Zero tensor components are dropped

Here all off-diagonal `Ydiag(i, j)` entries are defined as zero, so only the three diagonal vertices survive.


In [13]:
Ydiag = Parameter(
    "Ydiag",
    indices=(flavor, flavor),
    components={
        (1, 2): 0,
        (1, 3): 0,
        (2, 1): 0,
        (2, 3): 0,
        (3, 1): 0,
        (3, 2): 0,
    },
)
zero_model = Model(
    parameters=(Ydiag,),
    lagrangian_decl=S("gDiag") * psi.bar(f_mix) * Ydiag(f_mix, g_mix) * psi(g_mix) * phi,
)
zero_L = zero_model.lagrangian()

print("only diagonal signatures remain", [signature.names for signature in zero_L.vertex_signatures(flavor_expand=True)])
print()
show("compact Feynman rules", zero_L.feynman_rules())
show("expanded Feynman rules", zero_L.feynman_rules(flavor_expand=True))
print("e.bar, mu, Phi was dropped")
try:
    print(zero_L.feynman_rule(e.bar, mu, phi,  flavor_expand=True))
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


only diagonal signatures remain [('e.bar', 'e', 'Phi'), ('mu.bar', 'mu', 'Phi'), ('tau.bar', 'tau', 'Phi')]

compact Feynman rules
1 vertex signature(s)

Vertex: ('Psi.bar', 'Psi', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(f1,f2)

expanded Feynman rules
3 vertex signature(s)

Vertex: ('e.bar', 'e', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(1,1)

Vertex: ('mu.bar', 'mu', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(2,2)

Vertex: ('tau.bar', 'tau', 'Phi')
Rule: 1𝑖*gDiag*g(bis(4, i1),bis(4, i2))*Ydiag(3,3)

e.bar, mu, Phi was dropped
ValueError: No matching interaction terms for: e.bar, mu, Phi.
Available signatures:
  - e.bar, e, Phi
  - mu.bar, mu, Phi
  - tau.bar, tau, Phi


## 6. Gauge indices are not flavor-expanded

A quark class may carry `(flavor, color, spinor)`. Expansion removes only the flavor slot.


In [ ]:
fq = S("fq")
c = S("c")

flavor_q = flavor_index("FlavorQ", 2, prefix="fq")
q, (d, u) = flavor_family("q", ("d", "u"), flavor_q, extra_indices=(COLOR_FUND_INDEX,))

phi_q = scalar_field("PhiQ")

q_L = Model(
    fields=(q, phi_q),
    lagrangian_decl=S("gQ") * q.bar(fq, c) * q(fq, c) * phi_q,
).lagrangian()
expanded_terms = q_L._expanded_terms(flavor_expand=True)

print("expanded signatures", [signature.names for signature in q_L.vertex_signatures(flavor_expand=True)])
print("expanded field occurrences")
for term in expanded_terms:
    print(term.fields[0].field.name, term.fields[0].labels, term.fields[1].field.name, term.fields[1].labels)
print()
print("d.bar, d, PhiQ still carries a color identity tensor")
print(q_L.feynman_rule(d.bar, d, phi_q, simplify=True, include_delta=True, flavor_expand=True).format(show_namespaces=False, color_top_level_sum=False, color_builtin_symbols=False, multiplication_operator='*'))


expanded signatures [('d.bar', 'd', 'PhiQ'), ('u.bar', 'u', 'PhiQ')]
expanded field occurrences
d {'color_fund': c, 'spinor': i_decl_1} d {'color_fund': c, 'spinor': i_decl_1}
u {'color_fund': c, 'spinor': i_decl_1} u {'color_fund': c, 'spinor': i_decl_1}

d.bar, d, PhiQ still carries a color identity tensor
1𝑖*gQ*(2*𝜋)^d*g(bis(4,i1),bis(4,i2))*g(cof(3,c1),cof(3,c2))*Delta(q1+q2+q3)


## 7. Shared and independent flavor labels across classes

The same flavor label can synchronize two different classes. Different labels remain independent.


In [9]:
L_field, (eL, muL, tauL) = flavor_family("L", ("eL", "muL", "tauL"), flavor)
R_field, (eR, muR, tauR) = flavor_family("R", ("eR", "muR", "tauR"), flavor)
yLR = Parameter("yLR", indices=(flavor,), allow_summation=True)
shared_model = Model(
    fields=(L_field, R_field, phi),
    parameters=(yLR,),
    lagrangian_decl=yLR(f) * L_field.bar(f) * R_field(f) * phi,
)
shared_L = shared_model.lagrangian()

print("shared flavor label across L and R", [signature.names for signature in shared_L.vertex_signatures(flavor_expand=True)])
print()
print("tauL.bar, tauR, Phi")
print(shared_L.feynman_rule(tauL.bar, tauR, phi, simplify=True, include_delta=True, flavor_expand=True).format(show_namespaces=False, color_top_level_sum=False, color_builtin_symbols=False, multiplication_operator='*'))
print()

U_field, (u_phys, c_phys, t_phys) = flavor_family("U", ("u", "c", "t"), flavor)
D_field, (d_phys, s_phys, b_phys) = flavor_family("D", ("dQ", "sQ", "bQ"), flavor)
W = scalar_field("W")
V = Parameter("V", indices=(flavor, flavor))
independent_model = Model(
    fields=(U_field, D_field, W),
    parameters=(V,),
    lagrangian_decl=U_field.bar(f_mix) * V(f_mix, g_mix) * D_field(g_mix) * W,
)
independent_L = independent_model.lagrangian()

print("independent U[f] and D[g] labels", [signature.names for signature in independent_L.vertex_signatures(flavor_expand=True)])
print()
print("u.bar, sQ, W")
print(independent_L.feynman_rule(u_phys.bar, s_phys, W, simplify=True, include_delta=True, flavor_expand=True).format(show_namespaces=False, color_top_level_sum=False, color_builtin_symbols=False, multiplication_operator='*'))
print()
print("t.bar, bQ, W")
print(independent_L.feynman_rule(t_phys.bar, b_phys, W, simplify=True, include_delta=True, flavor_expand=True).format(show_namespaces=False, color_top_level_sum=False, color_builtin_symbols=False, multiplication_operator='*'))


shared flavor label across L and R [('eL.bar', 'eR', 'Phi'), ('muL.bar', 'muR', 'Phi'), ('tauL.bar', 'tauR', 'Phi')]

tauL.bar, tauR, Phi
1𝑖*(2*𝜋)^d*g(bis(4,i1),bis(4,i2))*Delta(q1+q2+q3)*yLR(3)

independent U[f] and D[g] labels [('c.bar', 'bQ', 'W'), ('c.bar', 'dQ', 'W'), ('c.bar', 'sQ', 'W'), ('t.bar', 'bQ', 'W'), ('t.bar', 'dQ', 'W'), ('t.bar', 'sQ', 'W'), ('u.bar', 'bQ', 'W'), ('u.bar', 'dQ', 'W'), ('u.bar', 'sQ', 'W')]

u.bar, sQ, W
1𝑖*(2*𝜋)^d*g(bis(4,i1),bis(4,i2))*Delta(q1+q2+q3)*V(1,2)

t.bar, bQ, W
1𝑖*(2*𝜋)^d*g(bis(4,i1),bis(4,i2))*Delta(q1+q2+q3)*V(3,3)


## 8. Representative error messages

The new checks try to fail early and clearly when declarations are inconsistent.


In [10]:
print("wrong member count")
try:
    Field(
        "BadPsi",
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S("badpsi"),
        conjugate_symbol=S("badpsibar"),
        indices=(flavor, SPINOR_INDEX),
        flavor_index=flavor,
        class_members=(e, mu),
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

psi_no_members = Field(
    "PsiNoMembers",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi_no_members"),
    conjugate_symbol=S("psi_no_members_bar"),
    indices=(flavor, SPINOR_INDEX),
    flavor_index=flavor,
)
missing_members_L = Model(
    fields=(psi_no_members, phi),
    lagrangian_decl=S("gMissing") * psi_no_members.bar(f) * psi_no_members(f) * phi,
).lagrangian()
print("flavor expansion requested but no class members are defined")
try:
    missing_members_L.vertex_signatures(flavor_expand=True)
except Exception as exc:
    print(type(exc).__name__ + ":", exc)
print()

plain_index = IndexType(
    "PlainGeneration",
    Representation.cof(3),
    kind="plain_generation",
    dimension=3,
    prefix="p",
)
print("flavor_index must use role=IndexRole.FLAVOR")
try:
    Field(
        "WrongRole",
        spin=Fraction(1, 2),
        self_conjugate=False,
        symbol=S("wrongrole"),
        conjugate_symbol=S("wrongrolebar"),
        indices=(plain_index, SPINOR_INDEX),
        flavor_index=plain_index,
        class_members=(e, mu, tau),
    )
except Exception as exc:
    print(type(exc).__name__ + ":", exc)


wrong member count
ValueError: Field 'BadPsi' declares 2 class member(s), but flavor index 'Flavor' has dimension 3.

flavor expansion requested but no class members are defined
ValueError: Cannot flavor-expand field 'PsiNoMembers' over index 'Flavor' because no class members are defined.

flavor_index must use role=IndexRole.FLAVOR
ValueError: Field 'WrongRole' declares flavor_index='PlainGeneration', but that index type is not marked with role=IndexRole.FLAVOR.
